<a href="https://colab.research.google.com/github/Leonavarro2287/Modelos2026/blob/main/Est_Descrip_y_graficos%2C_Normalizacion%2C_Ponderaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 📐 LÍNEA 1 – Estadística Descriptiva
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, base64, warnings
warnings.filterwarnings("ignore")

pd.options.display.float_format = '{:.4f}'.format

L1 = {"df": None, "df_stats": None, "df_criterios": None, "corr_matrix": None}

def _sep(texto=""):
    return widgets.HTML(f"<hr><b>{texto}</b>")

upload1     = widgets.FileUpload(accept=".xlsx,.xls,.csv", multiple=False,
                                  description="📂 Subir archivo", button_style="primary")
upload1_out = widgets.Output()
col_alt1    = widgets.Dropdown(description="Alternativas:", options=[],
                                style={"description_width":"110px"},
                                layout=widgets.Layout(width="320px"))
col_crit1   = widgets.SelectMultiple(description="Criterios:", options=[],
                                      layout=widgets.Layout(height="140px", width="380px"),
                                      style={"description_width":"80px"})
run1_btn    = widgets.Button(description="▶ Calcular estadísticas",
                              button_style="success",
                              layout=widgets.Layout(width="240px"))
run1_out    = widgets.Output()
dl1_btn     = widgets.Button(description="⬇ Descargar Excel",
                              button_style="info",
                              layout=widgets.Layout(width="200px"))
dl1_btn.layout.display = "none"
dl1_out     = widgets.Output()

def _load1(change):
    with upload1_out:
        clear_output()
        if not upload1.value: return
        key = list(upload1.value.keys())[0]
        fdata = upload1.value[key]["content"]
        try:
            df = pd.read_csv(io.BytesIO(fdata)) if key.endswith(".csv") else pd.read_excel(io.BytesIO(fdata))
            L1["df"] = df
            cols = list(df.columns)
            col_alt1.options = cols
            col_crit1.options = cols
            print(f"✅ {key}  |  {df.shape[0]} filas × {df.shape[1]} columnas")
            display(df.head())
        except Exception as e:
            print(f"❌ Error: {e}")

upload1.observe(_load1, names="value")

STATS_NOMBRES = [
    "Media", "Error típico", "Mediana", "Moda",
    "Desviación estándar", "CV (DS/Media)", "Varianza muestral",
    "Curtosis", "Coef. asimetría (Excel SKEW)",
    "Rango", "Mínimo", "Máximo", "Máx/Mín",
    "Suma", "Cuenta", "Módulo"
]

def _calcular_stats(serie):
    s = pd.to_numeric(serie, errors="coerce").dropna()
    n = len(s)
    media    = s.mean()
    err_tip  = stats.sem(s)
    mediana  = s.median()
    try:    moda = s.mode().iloc[0]
    except: moda = np.nan
    ds       = s.std(ddof=1)
    cv       = ds / media if media != 0 else np.nan
    varianza = s.var(ddof=1)
    kurt     = s.kurt()
    if n > 2 and ds != 0:
        asim = (n / ((n-1) * (n-2))) * (((s - media) / ds)**3).sum()
    else:
        asim = np.nan
    rango   = s.max() - s.min()
    minimo  = s.min()
    maximo  = s.max()
    max_min = maximo / minimo if minimo != 0 else np.nan
    suma    = s.sum()
    cuenta  = n
    modulo  = np.sqrt((s**2).sum())
    return [media, err_tip, mediana, moda, ds, cv, varianza,
            kurt, asim, rango, minimo, maximo, max_min, suma, cuenta, modulo]

def _run1(b):
    with run1_out:
        clear_output()
        df = L1["df"]
        if df is None: print("❌ Cargá un archivo primero."); return
        crit_cols = list(col_crit1.value)
        if not crit_cols: print("❌ Seleccioná al menos un criterio."); return
        df_c = df[crit_cols].apply(pd.to_numeric, errors="coerce")
        L1["df_criterios"] = df_c
        filas = {col: _calcular_stats(df_c[col]) for col in crit_cols}
        df_stats = pd.DataFrame(filas, index=STATS_NOMBRES)
        L1["df_stats"] = df_stats
        display(HTML("<b>Estadísticas descriptivas</b>"))
        display(df_stats.round(4))

        # ---------- NUEVO: Matriz de correlación ----------
        if len(crit_cols) >= 2:
            corr_matrix = df_c[crit_cols].corr()
            L1["corr_matrix"] = corr_matrix
            display(HTML("<br><b>Matriz de correlación de Pearson</b>"))
            display(corr_matrix.round(4))
        else:
            L1["corr_matrix"] = None
            print("\n⚠️ Se necesitan al menos 2 criterios para la matriz de correlación.")
        # -------------------------------------------------

        # Boxplots
        display(HTML("<b>Boxplots</b>"))
        n_crit = len(crit_cols)
        colors = plt.cm.tab10.colors
        fig, axes = plt.subplots(1, n_crit, figsize=(max(4*n_crit, 6), 5), squeeze=False)
        for idx, col in enumerate(crit_cols):
            ax = axes[0][idx]
            bp = ax.boxplot(df_c[col].dropna(), patch_artist=True,
                            medianprops=dict(color="white", linewidth=2))
            bp["boxes"][0].set_facecolor(colors[idx % 10])
            ax.set_title(col, fontsize=11, fontweight="bold")
            ax.set_xticks([])
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
            ax.grid(axis="y", linestyle="--", alpha=0.5)
        plt.suptitle("Boxplots por criterio", fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout(); plt.show()
        # Histogramas
        display(HTML("<b>Histogramas</b>"))
        fig2, axes2 = plt.subplots(1, n_crit, figsize=(max(4*n_crit, 6), 4), squeeze=False)
        for idx, col in enumerate(crit_cols):
            ax = axes2[0][idx]
            ax.hist(df_c[col].dropna(), bins="auto", color=colors[idx % 10],
                    edgecolor="white", alpha=0.85)
            ax.set_title(col, fontsize=11, fontweight="bold")
            ax.set_xlabel("Valor", fontsize=9); ax.set_ylabel("Frecuencia", fontsize=9)
            ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
            ax.grid(axis="y", linestyle="--", alpha=0.4)
        plt.suptitle("Histogramas por criterio", fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout(); plt.show()
        dl1_btn.layout.display = ""

run1_btn.on_click(_run1)

def _download1(b):
    with dl1_out:
        clear_output()
        df_stats = L1.get("df_stats")
        df_c     = L1.get("df_criterios")
        corr_mat = L1.get("corr_matrix")
        if df_stats is None: print("❌ Calculá primero."); return
        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine="openpyxl") as writer:
            df_stats.round(4).to_excel(writer, sheet_name="Estadisticas", startrow=2)
            ws1 = writer.sheets["Estadisticas"]
            ws1.cell(row=1, column=1,
                     value="ESTADÍSTICAS DESCRIPTIVAS – Modelos de Decisión | Unidad 3")
            if df_c is not None:
                df_c.round(4).to_excel(writer, sheet_name="Datos_criterios", startrow=2)
                ws2 = writer.sheets["Datos_criterios"]
                ws2.cell(row=1, column=1, value="DATOS ORIGINALES – Criterios seleccionados")
            if corr_mat is not None:
                corr_mat.round(4).to_excel(writer, sheet_name="Matriz_correlacion", startrow=2)
                ws3 = writer.sheets["Matriz_correlacion"]
                ws3.cell(row=1, column=1, value="MATRIZ DE CORRELACIÓN (Pearson)")
        buf.seek(0)
        b64 = base64.b64encode(buf.getvalue()).decode()
        display(HTML(f'<a download="estadisticas.xlsx" '
                     f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">'
                     f'⬇ Descargar estadisticas.xlsx</a>'))

dl1_btn.on_click(_download1)

display(widgets.HTML("<h3>📐 LÍNEA 1 – Estadística Descriptiva</h3>"))
display(_sep("1. Cargar archivo (.xlsx / .xls / .csv)"))
display(upload1, upload1_out)
display(_sep("2. Seleccionar columnas"))
display(widgets.HBox([col_alt1, col_crit1]))
display(_sep("3. Calcular"))
display(run1_btn, run1_out)
display(dl1_btn, dl1_out)

In [ ]:
#@title 🔢 LÍNEA 2 – Normalización (con transformación de mínimo a máximo)
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, base64, warnings
warnings.filterwarnings("ignore")

pd.options.display.float_format = '{:.4f}'.format

# ------------------------------------------------------------
# Funciones de normalización (todas implementadas con fórmulas)
# ------------------------------------------------------------
def _norm_fraccion_max(df):
    return df.div(df.max())

def _norm_fraccion_suma(df):
    return df.div(df.sum())

def _norm_fraccion_rango(df):
    rango = df.max() - df.min()
    return (df - df.min()).div(rango.replace(0, np.nan))

def _norm_vector(df):
    return df.div(np.sqrt((df**2).sum()))

def _norm_zscore(df):
    return (df - df.mean()) / df.std(ddof=0)

def _norm_ideal_ref(df, metas_rim):
    """
    Normalización por referencia ideal (RIM).
    metas_rim: dict {criterio: (C, D)} con los límites del rango ideal.
    A y B se toman como el mínimo y máximo de cada columna.
    """
    result = df.copy()
    for col in df.columns:
        c, d = metas_rim[col]
        a = df[col].min()
        b = df[col].max()
        x = df[col].values
        v = np.where(x < c,
                     np.maximum(0, (x - a) / (c - a)) if c != a else 0,
                     np.where(x <= d,
                              1.0,
                              np.maximum(0, (b - x) / (b - d)) if b != d else 0))
        result[col] = v
    return result

NORM_METODOS = {
    "Fracción del máximo":  _norm_fraccion_max,
    "Fracción de la suma":  _norm_fraccion_suma,
    "Fracción del rango":   _norm_fraccion_rango,
    "Del vector":           _norm_vector,
    "Z-score":              _norm_zscore,
    "Ideal de referencia":  None,   # se maneja aparte porque necesita parámetros
    "(Sin normalizar)":     lambda df: df.copy(),
}

# ------------------------------------------------------------
# Transformación de criterios de mínimo a máximo
# Paso 1: recíproca = 1 / valor
# Paso 2: dividir cada recíproco entre el máximo de los recíprocos
# Eso produce: r_ij = min(original) / valor_original
# ------------------------------------------------------------
def _transformar_minimo_a_maximo(df, min_cols):
    """
    Transforma criterios de mínimo a máximo usando: min(columna) / valor_actual.
    """
    df_trans = df.copy()
    for col in min_cols:
        if col in df_trans.columns:
            min_val = df_trans[col].min()
            # Se aplica la fórmula directa: min / valor
            # El replace evita la división por cero si hay valores 0 en los datos
            df_trans[col] = min_val / df_trans[col].replace(0, 1e-12)
    return df_trans
# ------------------------------------------------------------
# Widgets y datos globales
# ------------------------------------------------------------
L2 = {"df": None, "df_norm": None, "df_show": None, "df_trans": None}

def _sep(texto=""):
    return widgets.HTML(f"<hr><b>{texto}</b>")

upload2     = widgets.FileUpload(accept=".xlsx,.xls,.csv", multiple=False,
                                  description="📂 Subir archivo", button_style="primary")
upload2_out = widgets.Output()
col_alt2    = widgets.Dropdown(description="Alternativas:", options=[],
                                style={"description_width":"110px"},
                                layout=widgets.Layout(width="320px"))
col_crit2   = widgets.SelectMultiple(description="Criterios:", options=[],
                                      layout=widgets.Layout(height="140px", width="380px"),
                                      style={"description_width":"80px"})
col_min2    = widgets.SelectMultiple(description="Criterios de mínimo:",
                                     options=[],
                                     layout=widgets.Layout(height="100px", width="380px"),
                                     style={"description_width":"120px"})
norm_method2 = widgets.Dropdown(
    description="Normalización:",
    options=list(NORM_METODOS.keys()),
    value="Fracción de la suma",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="360px"))
run2_btn    = widgets.Button(description="▶ Normalizar", button_style="success",
                              layout=widgets.Layout(width="200px"))
run2_out    = widgets.Output()
dl2_btn     = widgets.Button(description="⬇ Descargar matriz normalizada",
                              button_style="info",
                              layout=widgets.Layout(width="270px"))
dl2_btn.layout.display = "none"
dl2_out     = widgets.Output()

# Contenedor para los parámetros C y D del RIM
rim_box = widgets.VBox([])
rim_inputs = {}

def _crear_rim_inputs(criterios):
    """Crea o actualiza los widgets de C y D para los criterios seleccionados."""
    global rim_inputs
    children = []
    rim_inputs = {}
    for crit in criterios:
        if L2["df"] is not None and crit in L2["df"].columns:
            col_data = L2["df"][crit].dropna()
            sugerido_c = col_data.quantile(0.75) if len(col_data) else 0.0
            sugerido_d = col_data.max() if len(col_data) else 0.0
        else:
            sugerido_c, sugerido_d = 0.0, 1.0
        w_c = widgets.BoundedFloatText(value=round(sugerido_c, 4),
                                       min=-1e6, max=1e6, step=0.01,
                                       description=f'{crit[:15]} C:',
                                       layout=widgets.Layout(width='280px'))
        w_d = widgets.BoundedFloatText(value=round(sugerido_d, 4),
                                       min=-1e6, max=1e6, step=0.01,
                                       description='D:',
                                       layout=widgets.Layout(width='280px'))
        rim_inputs[crit] = (w_c, w_d)
        children.append(widgets.HBox([w_c, w_d]))
    rim_box.children = children

def _actualizar_visibilidad_rim(*args):
    if norm_method2.value == "Ideal de referencia":
        _crear_rim_inputs(col_crit2.value)
        rim_box.layout.display = ""
    else:
        rim_box.layout.display = "none"

norm_method2.observe(_actualizar_visibilidad_rim, names="value")
col_crit2.observe(_actualizar_visibilidad_rim, names="value")

# ------------------------------------------------------------
# Actualizar opciones de criterios de mínimo según los criterios seleccionados
# ------------------------------------------------------------
def _actualizar_min_options(change=None):
    crit_seleccionados = list(col_crit2.value)
    actuales = list(col_min2.value)
    nuevos_validos = [c for c in actuales if c in crit_seleccionados]
    col_min2.options = crit_seleccionados
    col_min2.value = nuevos_validos

col_crit2.observe(_actualizar_min_options, names="value")

# ------------------------------------------------------------
# Función auxiliar para generar enlace de descarga en HTML
# ------------------------------------------------------------
def _descargar_enlace(df, nombre_archivo, titulo_hoja, metodo):
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as writer:
        df.round(4).to_excel(writer, sheet_name=titulo_hoja, index=False, startrow=2)
        ws = writer.sheets[titulo_hoja]
        ws.cell(row=1, column=1,
                value=f"{titulo_hoja.upper()} – Método: {metodo} | Modelos de Decisión | Unidad 3")
    buf.seek(0)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return HTML(f'<a download="{nombre_archivo}" '
                f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">'
                f'⬇ Descargar {nombre_archivo}</a>')

# ------------------------------------------------------------
# Carga de archivo
# ------------------------------------------------------------
def _load2(change):
    with upload2_out:
        clear_output()
        if not upload2.value: return
        key = list(upload2.value.keys())[0]
        fdata = upload2.value[key]["content"]
        try:
            df = pd.read_csv(io.BytesIO(fdata)) if key.endswith(".csv") else pd.read_excel(io.BytesIO(fdata))
            L2["df"] = df
            cols = list(df.columns)
            col_alt2.options = cols
            col_crit2.options = cols
            col_min2.options = []
            col_min2.value = []
            print(f"✅ {key}  |  {df.shape[0]} filas × {df.shape[1]} columnas")
            display(df.head())
            _actualizar_visibilidad_rim()
        except Exception as e:
            print(f"❌ Error: {e}")

upload2.observe(_load2, names="value")

# ------------------------------------------------------------
# Ejecutar normalización
# ------------------------------------------------------------
def _run2(b):
    with run2_out:
        clear_output()
        df = L2["df"]
        if df is None: print("❌ Cargá un archivo primero."); return
        crit_cols = list(col_crit2.value)
        if not crit_cols: print("❌ Seleccioná al menos un criterio."); return
        metodo = norm_method2.value
        min_cols = list(col_min2.value)

        # Extraer datos numéricos
        df_c = df[crit_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

        # --------------------------------------------
        # 1. Transformar criterios de mínimo a máximo
        #    (recíproca → dividir por el máximo de los recíprocos)
        # --------------------------------------------
        df_trans = df_c.copy()
        if min_cols:
            df_trans = _transformar_minimo_a_maximo(df_c, min_cols)
            print("✅ Se transformaron a máximo los criterios:", ", ".join(min_cols))
        else:
            print("ℹ️ No se marcaron criterios de mínimo.")

        # Construir df_show para la matriz transformada (con alternativas)
        alt_col = col_alt2.value
        if alt_col and alt_col in df.columns:
            df_trans_show = df_trans.copy()
            df_trans_show.insert(0, alt_col, df[alt_col].values)
        else:
            df_trans_show = df_trans.copy()

        # Mostrar matriz transformada
        display(HTML("<b>📊 Matriz Transformada (mínimos convertidos a máximo)</b>"))
        display(df_trans_show.round(4))

        # Enlace de descarga para la matriz transformada
        display(_descargar_enlace(df_trans_show, "matriz_transformada.xlsx",
                                  "Transformada", metodo))

        display(HTML("<hr>"))

        # --------------------------------------------
        # 2. Aplicar normalización elegida sobre df_trans
        # --------------------------------------------
        try:
            if metodo == "Ideal de referencia":
                # Recoger C y D del usuario (basados en df_trans)
                rim_dict = {}
                for crit in crit_cols:
                    wc, wd = rim_inputs[crit]
                    rim_dict[crit] = (wc.value, wd.value)
                df_norm = _norm_ideal_ref(df_trans, rim_dict)
            else:
                df_norm = NORM_METODOS[metodo](df_trans)
        except Exception as e:
            print(f"❌ Error al normalizar: {e}"); return

        # Construir df_show para la matriz normalizada
        if alt_col and alt_col in df.columns:
            df_norm_show = df_norm.copy()
            df_norm_show.insert(0, alt_col, df[alt_col].values)
        else:
            df_norm_show = df_norm.copy()

        L2["df_norm"] = df_norm
        L2["df_show"] = df_norm_show
        L2["df_trans"] = df_trans

        display(HTML(f"<b>🔢 Matriz Normalizada – {metodo}</b>"))
        display(df_norm_show.round(4))
        dl2_btn.layout.display = ""

run2_btn.on_click(_run2)

# ------------------------------------------------------------
# Descarga de la matriz normalizada (botón)
# ------------------------------------------------------------
def _download2(b):
    with dl2_out:
        clear_output()
        df_show = L2.get("df_show")
        if df_show is None: print("❌ Normalizá primero."); return
        metodo = norm_method2.value
        display(_descargar_enlace(df_show, "matriz_normalizada.xlsx",
                                  "Normalizada", metodo))

dl2_btn.on_click(_download2)

# ------------------------------------------------------------
# Mostrar interfaz
# ------------------------------------------------------------
display(widgets.HTML("<h3>🔢 LÍNEA 2 – Normalización <small>con transformación exacta de mínimo a máximo</small></h3>"))
display(_sep("1. Cargar archivo"))
display(upload2, upload2_out)
display(_sep("2. Seleccionar columnas"))
display(widgets.HBox([col_alt2, col_crit2]))
display(_sep("Indicar criterios de mínimo (se aplicará recíproca + división por el máximo recíproco)"))
display(col_min2)
display(_sep("3. Elegir método de normalización"))
display(norm_method2)
display(_sep("Parámetros para Ideal de referencia (RIM)"))
display(rim_box)
display(_sep("4. Normalizar"))
display(run2_btn, run2_out)
display(dl2_btn, dl2_out)

In [ ]:
#@title ⚖️ LÍNEA 3 – Calculadora de Ponderaciones
import numpy as np
import pandas as pd
from itertools import combinations
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, warnings
warnings.filterwarnings("ignore")

pd.options.display.float_format = '{:.4f}'.format

L3 = {"df": None}

def _sep(texto=""):
    return widgets.HTML(f"<hr><b>{texto}</b>")

# ------------------------------ Funciones de ponderación ------------------------------
def pesos_uniformes(df, **kw):
    n = df.shape[1]
    return pd.Series({c: 1/n for c in df.columns})

def pesos_desv_estandar(df, **kw):
    s = df.std(ddof=0); return s / s.sum()

def pesos_coef_variacion(df, **kw):
    cv = df.std(ddof=0).abs() / df.mean().abs().replace(0, np.nan)
    cv = cv.fillna(0); tot = cv.sum()
    return cv / tot if tot != 0 else pd.Series({c: np.nan for c in df.columns})

def pesos_entropia(df, **kw):
    r = df.div(df.sum()).fillna(0)
    r_safe = r.replace(0, 1e-12)
    m = r.shape[0]; k = 1 / np.log(m) if m > 1 else 1
    ej = -k * (r_safe * np.log(r_safe)).sum()
    dj = 1 - ej; tot = dj.sum()
    return dj / tot if tot != 0 else pd.Series({c: np.nan for c in df.columns})

def pesos_critic(df, **kw):
    r_norm = (df - df.min()) / (df.max() - df.min()).replace(0, np.nan)
    r_norm = r_norm.fillna(0); s = r_norm.std(ddof=0); corr = r_norm.corr()
    wj = {j: s[j] * sum(1 - corr.loc[j, k] for k in df.columns) for j in df.columns}
    w = pd.Series(wj); tot = w.sum()
    return w / tot if tot != 0 else pd.Series({c: np.nan for c in df.columns})

def pesos_ordenacion_simple(df, orden=None, **kw):
    if orden is None: orden = {c: i+1 for i, c in enumerate(df.columns)}
    s = pd.Series(orden); tot = s.sum()
    return s / tot if tot != 0 else pd.Series({c: np.nan for c in df.columns})

def pesos_tasacion_simple(df, puntuaciones=None, **kw):
    if puntuaciones is None: puntuaciones = {c: 1 for c in df.columns}
    s = pd.Series(puntuaciones); tot = s.sum()
    return s / tot if tot != 0 else pd.Series({c: np.nan for c in df.columns})

def pesos_ahp(df, matriz_comparacion=None, **kw):
    cols = list(df.columns); n = len(cols)
    M = matriz_comparacion if matriz_comparacion is not None else pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    M_norm = M.div(M.sum()); w = M_norm.mean(axis=1)
    return w / w.sum()

PONDERACIONES3 = {
    "Uniforme":             pesos_uniformes,
    "Desviación estándar":  pesos_desv_estandar,
    "Coef. de variación":   pesos_coef_variacion,
    "Entropía":             pesos_entropia,
    "CRITIC":               pesos_critic,
    "Ordenación simple":    pesos_ordenacion_simple,
    "Tasación simple":      pesos_tasacion_simple,
    "AHP":                  pesos_ahp,
}

# ------------------------------ Normalizaciones ------------------------------
def _norm_fraccion_max(df): return df.div(df.max())
def _norm_fraccion_suma(df): return df.div(df.sum())
def _norm_fraccion_rango(df):
    rango = df.max() - df.min()
    return (df - df.min()).div(rango.replace(0, np.nan))
def _norm_vector(df): return df.div(np.sqrt((df**2).sum()))
def _norm_zscore(df): return (df - df.mean()) / df.std(ddof=0)

def _norm_ideal_ref(df, metas_rim):
    """RIM: metas_rim = {criterio: (C, D)}"""
    result = df.copy()
    for col in df.columns:
        c, d = metas_rim[col]
        a = df[col].min()
        b = df[col].max()
        x = df[col].values
        v = np.where(x < c,
                     np.maximum(0, (x - a) / (c - a)) if c != a else 0,
                     np.where(x <= d,
                              1.0,
                              np.maximum(0, (b - x) / (b - d)) if b != d else 0))
        result[col] = v
    return result

NORM_FUNC3 = {
    "(Sin normalizar)":     lambda d: d.copy(),
    "Fracción del máximo":  _norm_fraccion_max,
    "Fracción de la suma":  _norm_fraccion_suma,
    "Fracción del rango":   _norm_fraccion_rango,
    "Del vector":           _norm_vector,
    "Z-score":              _norm_zscore,
    "Ideal de referencia":  None,  # se maneja aparte
}

# ------------------------------ Widgets de interfaz ------------------------------
upload3     = widgets.FileUpload(accept=".xlsx,.xls,.csv", multiple=False,
                                  description="📂 Subir archivo", button_style="primary")
upload3_out = widgets.Output()
col_alt3    = widgets.Dropdown(description="Alternativas:", options=[],
                                style={"description_width":"110px"},
                                layout=widgets.Layout(width="320px"))
col_crit3   = widgets.SelectMultiple(description="Criterios:", options=[],
                                      layout=widgets.Layout(height="140px", width="380px"),
                                      style={"description_width":"80px"})
norm3_method = widgets.Dropdown(
    description="Normalización previa:",
    options=list(NORM_FUNC3.keys()),
    value="(Sin normalizar)",
    style={"description_width":"150px"}, layout=widgets.Layout(width="400px"))

# Parámetros RIM (C y D) – ocultos hasta que se elija "Ideal de referencia"
rim3_box = widgets.VBox([])
rim3_inputs = {}  # {criterio: (widget_c, widget_d)}

pond3_select = widgets.SelectMultiple(
    options=list(PONDERACIONES3.keys()), value=["Uniforme"],
    description="Ponderaciones:",
    layout=widgets.Layout(height="200px", width="340px"),
    style={"description_width":"120px"})
ahp3_out = widgets.Output()
ahp3_panel = widgets.VBox([
    widgets.HTML("<b>Matriz AHP (comparaciones pareadas)</b><br>"
                 "<small>1 = igual importancia, 9 = extremadamente más importante.</small>"),
    ahp3_out])
ahp3_panel.layout.display = "none"
ord3_out = widgets.Output()
ord3_panel = widgets.VBox([
    widgets.HTML("<b>Ordenación Simple – Rangos</b><br>"
                 "<small>1 = menos importante, n = más importante.</small>"),
    ord3_out])
ord3_panel.layout.display = "none"
tas3_out = widgets.Output()
tas3_panel = widgets.VBox([
    widgets.HTML("<b>Tasación Simple – Puntajes</b>"),
    tas3_out])
tas3_panel.layout.display = "none"
run3_btn = widgets.Button(description="▶ Calcular pesos", button_style="success",
                           layout=widgets.Layout(width="200px"))
run3_out = widgets.Output()

# ------------------------------ Funciones de renderizado dinámico ------------------------------
def _render3_rim():
    """Crea campos C y D para los criterios actuales."""
    df = L3["df"]
    crit_cols = list(col_crit3.value) if col_crit3.value else []
    if df is None or not crit_cols:
        rim3_box.children = [widgets.HTML("<small>Seleccioná criterios para definir C y D.</small>")]
        return
    global rim3_inputs
    rim3_inputs = {}
    children = []
    for crit in crit_cols:
        col_data = pd.to_numeric(df[crit], errors="coerce").dropna()
        if len(col_data) > 0:
            sugerido_c = col_data.quantile(0.75)
            sugerido_d = col_data.max()
        else:
            sugerido_c, sugerido_d = 0.0, 1.0
        w_c = widgets.BoundedFloatText(value=round(sugerido_c, 4),
                                       min=-1e6, max=1e6, step=0.01,
                                       description=f'{crit[:15]} C:',
                                       layout=widgets.Layout(width='280px'))
        w_d = widgets.BoundedFloatText(value=round(sugerido_d, 4),
                                       min=-1e6, max=1e6, step=0.01,
                                       description='D:',
                                       layout=widgets.Layout(width='280px'))
        rim3_inputs[crit] = (w_c, w_d)
        children.append(widgets.HBox([w_c, w_d]))
    rim3_box.children = children

def _render3_ahp():
    df = L3["df"]
    if df is None or not L3.get("crit_cols"): return
    cols = L3["crit_cols"]; n = len(cols)
    sliders = {}; rows = []
    for (i, j) in combinations(range(n), 2):
        s = widgets.FloatText(value=1.0, step=0.5,
                              description=f"{cols[i]} vs {cols[j]}:",
                              style={"description_width":"260px"},
                              layout=widgets.Layout(width="380px"))
        sliders[(i, j)] = s; rows.append(s)
    L3["ahp_widgets"] = (sliders, cols, n)
    with ahp3_out:
        clear_output(); display(widgets.VBox(rows))

def _render3_ord():
    df = L3["df"]
    if df is None or not L3.get("crit_cols"): return
    cols = L3["crit_cols"]; wmap = {}; rows = []
    for c in cols:
        w = widgets.IntText(value=1, description=f"{c}:",
                            style={"description_width":"200px"},
                            layout=widgets.Layout(width="320px"))
        wmap[c] = w; rows.append(w)
    L3["ord_widgets"] = wmap
    with ord3_out:
        clear_output(); display(widgets.VBox(rows))

def _render3_tas():
    df = L3["df"]
    if df is None or not L3.get("crit_cols"): return
    cols = L3["crit_cols"]; wmap = {}; rows = []
    for c in cols:
        w = widgets.FloatText(value=1.0, description=f"{c}:",
                              style={"description_width":"200px"},
                              layout=widgets.Layout(width="320px"))
        wmap[c] = w; rows.append(w)
    L3["tas_widgets"] = wmap
    with tas3_out:
        clear_output(); display(widgets.VBox(rows))

def _update3_panels(change=None):
    sel = pond3_select.value
    ahp3_panel.layout.display = "" if "AHP" in sel else "none"
    ord3_panel.layout.display = "" if "Ordenación simple" in sel else "none"
    tas3_panel.layout.display = "" if "Tasación simple" in sel else "none"
    # Mostrar/ocultar panel RIM según normalización
    if norm3_method.value == "Ideal de referencia":
        _render3_rim()
        rim3_box.layout.display = ""
    else:
        rim3_box.layout.display = "none"

pond3_select.observe(_update3_panels, names="value")
norm3_method.observe(_update3_panels, names="value")

def _load3(change):
    with upload3_out:
        clear_output()
        if not upload3.value: return
        key = list(upload3.value.keys())[0]
        fdata = upload3.value[key]["content"]
        try:
            df = pd.read_csv(io.BytesIO(fdata)) if key.endswith(".csv") else pd.read_excel(io.BytesIO(fdata))
            L3["df"] = df
            cols = list(df.columns)
            col_alt3.options = cols; col_crit3.options = cols
            print(f"✅ {key}  |  {df.shape[0]} filas × {df.shape[1]} columnas")
            display(df.head())
        except Exception as e:
            print(f"❌ Error: {e}")

upload3.observe(_load3, names="value")

def _on_crit3_change(change):
    L3["crit_cols"] = list(col_crit3.value)
    _render3_ahp(); _render3_ord(); _render3_tas()
    if norm3_method.value == "Ideal de referencia":
        _render3_rim()

col_crit3.observe(_on_crit3_change, names="value")

# ------------------------------ Ejecución principal ------------------------------
def _run3(b):
    with run3_out:
        clear_output()
        df = L3["df"]
        if df is None: print("❌ Cargá un archivo primero."); return
        crit_cols = list(col_crit3.value)
        if not crit_cols: print("❌ Seleccioná al menos un criterio."); return
        ponds_sel = list(pond3_select.value)
        if not ponds_sel: print("❌ Seleccioná al menos un método de ponderación."); return
        df_c = df[crit_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        metodo_norm = norm3_method.value

        # Aplicar normalización
        if metodo_norm == "Ideal de referencia":
            if not rim3_inputs:
                print("❌ No se definieron parámetros C y D para RIM. Seleccioná 'Ideal de referencia' y completá los campos.")
                return
            metas = {}
            for crit in crit_cols:
                wc, wd = rim3_inputs[crit]
                metas[crit] = (wc.value, wd.value)
            df_norm = _norm_ideal_ref(df_c, metas)
        else:
            df_norm = NORM_FUNC3[metodo_norm](df_c)
        df_norm = df_norm.fillna(0)

        # 🔹 Mostrar la matriz normalizada antes de ponderar 🔹
        display(HTML(f"<b>Matriz normalizada</b> <small>({metodo_norm})</small>"))
        alt_col = col_alt3.value
        if alt_col and alt_col in df.columns:
            display_df = df_norm.copy()
            display_df.insert(0, alt_col, df[alt_col].values)
        else:
            display_df = df_norm.copy()
        display(display_df.round(4))
        display(HTML("<hr>"))  # separador visual

        # Parámetros adicionales para ponderaciones
        kwargs_extra = {}
        if "AHP" in ponds_sel and "ahp_widgets" in L3:
            sliders, cols, n = L3["ahp_widgets"]
            M = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
            for (i, j), s in sliders.items():
                val = s.value if s.value > 0 else 1.0
                M.iloc[i, j] = val; M.iloc[j, i] = 1 / val
            kwargs_extra["AHP"] = {"matriz_comparacion": M}
        if "Ordenación simple" in ponds_sel and "ord_widgets" in L3:
            kwargs_extra["Ordenación simple"] = {"orden": {c: w.value for c, w in L3["ord_widgets"].items()}}
        if "Tasación simple" in ponds_sel and "tas_widgets" in L3:
            kwargs_extra["Tasación simple"] = {"puntuaciones": {c: w.value for c, w in L3["tas_widgets"].items()}}

        display(HTML(f"<b>Pesos calculados</b>"))
        rows_pesos = {}
        for met in ponds_sel:
            fn = PONDERACIONES3[met]; kw = kwargs_extra.get(met, {})
            try:
                w = fn(df_norm, **kw)
                rows_pesos[met] = {c: round(float(w[c]), 4) for c in crit_cols}
            except Exception as e:
                rows_pesos[met] = {c: f"Error: {e}" for c in crit_cols}
        df_pesos = pd.DataFrame(rows_pesos).T
        df_pesos.index.name = "Método"; df_pesos.columns.name = "Criterio"
        try:
            df_pesos["∑ pesos"] = df_pesos.astype(float).sum(axis=1).round(4)
        except Exception:
            pass
        display(df_pesos)
        print("✅ Suma de pesos ≈ 1 por método (verificar columna ∑ pesos)")

run3_btn.on_click(_run3)

# ------------------------------ Mostrar interfaz ------------------------------
display(widgets.HTML("<h3>⚖️ LÍNEA 3 – Calculadora de Ponderaciones <small>solo muestra pesos</small></h3>"))
display(_sep("1. Cargar archivo"))
display(upload3, upload3_out)
display(_sep("2. Seleccionar columnas"))
display(widgets.HBox([col_alt3, col_crit3]))
display(_sep("3. Normalización previa (opcional)"))
display(norm3_method)
display(_sep("Parámetros para Ideal de referencia (RIM)"))
display(rim3_box)
display(_sep("4. Elegir métodos de ponderación (Ctrl+clic para varios)"))
display(pond3_select)
display(_sep("5. Parámetros adicionales (si aplica)"))
display(ahp3_panel, ord3_panel, tas3_panel)
display(_sep("6. Calcular"))
display(run3_btn, run3_out)

In [ ]:
#@title 📊 LÍNEA 4 – Agregación Multicriterio (Suma Ponderada y Media Geométrica Ponderada)
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, base64, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

pd.options.display.float_format = '{:.4f}'.format

# ------------------------------------------------------------
# Variables globales
# ------------------------------------------------------------
L4 = {"df_norm": None, "df_show": None, "pesos": None, "df_agreg": None, "matriz_transform": None,
      "crit_cols": None, "alternativas": None}

# Intentar recuperar L2 si existe (para cargar de ahí)
try:
    L2
except NameError:
    L2 = {"df_show": None}

# ------------------------------------------------------------
# Funciones de agregación
# ------------------------------------------------------------
def suma_ponderada(matriz, pesos):
    """U(a_i) = Σ_j w_j * r_ij"""
    return matriz.dot(pesos)

def media_geometrica_ponderada(matriz, pesos, return_matriz=True):
    """
    U(a_i) = ( Π_j (r_ij ^ w_j) ) ^ (1 / n)
    donde n = número de criterios
    Si return_matriz=True, devuelve (resultados, matriz_transformada)
    """
    n_crit = matriz.shape[1]                     # número de criterios
    matriz_potencia = np.power(matriz, pesos)      # shape (m, n)
    producto = np.prod(matriz_potencia, axis=1)    # Π_j r_ij^w_j
    resultado = producto ** (1.0 / n_crit)         # raíz n-ésima
    if return_matriz:
        return resultado, matriz_potencia
    else:
        return resultado

# ------------------------------------------------------------
# Elementos comunes de la interfaz
# ------------------------------------------------------------
def _sep(texto=""):
    return widgets.HTML(f"<hr><b>{texto}</b>")

# ------------------------------------------------------------
# Widgets de la interfaz
# ------------------------------------------------------------
origen_data = widgets.RadioButtons(
    options=["📂 Cargar archivo de matriz normalizada", "📎 Usar matriz normalizada de L2"],
    value="📂 Cargar archivo de matriz normalizada",
    description="Origen de datos:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="auto")
)

# Widgets para carga de archivo
upload4 = widgets.FileUpload(accept=".xlsx,.xls,.csv", multiple=False,
                              description="📂 Subir matriz normalizada", button_style="primary")
upload4_out = widgets.Output()

col_alt4 = widgets.Dropdown(description="Alternativas:", options=[],
                             style={"description_width": "110px"},
                             layout=widgets.Layout(width="320px"))
col_crit4 = widgets.SelectMultiple(description="Criterios:", options=[],
                                    layout=widgets.Layout(height="140px", width="380px"),
                                    style={"description_width": "80px"})

# Contenedor dinámico para los pesos
pesos_box = widgets.VBox([])
pesos_widgets = {}  # {criterio: FloatText}

btn_generar_pesos = widgets.Button(description="🔄 Actualizar campos de pesos",
                                   button_style="info",
                                   layout=widgets.Layout(width="250px"))

# Método de agregación
metodo_agreg4 = widgets.Dropdown(
    options=["Suma ponderada", "Media geométrica ponderada"],
    value="Suma ponderada",
    description="Método:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="330px")
)

# Botón de ejecución y salidas
run4_btn = widgets.Button(description="▶ Calcular agregación", button_style="success",
                          layout=widgets.Layout(width="240px"))
run4_out = widgets.Output()
dl4_btn = widgets.Button(description="⬇ Descargar resultados", button_style="info",
                         layout=widgets.Layout(width="220px"))
dl4_btn.layout.display = "none"
dl4_out = widgets.Output()

# ------------------------------------------------------------
# Función para crear/actualizar los campos de pesos
# ------------------------------------------------------------
def _actualizar_campos_pesos(criterios):
    """Crea FloatText para cada criterio con valor inicial 1/n."""
    global pesos_widgets
    pesos_widgets.clear()
    n = len(criterios)
    valor_inicial = 1.0 / n if n > 0 else 0
    children = []
    for crit in criterios:
        w = widgets.FloatText(value=round(valor_inicial, 4),
                              min=0, max=1e6, step=0.01,
                              description=f'{crit[:15]}:',
                              layout=widgets.Layout(width='300px'))
        pesos_widgets[crit] = w
        children.append(w)
    pesos_box.children = children

def _on_generar_pesos_click(b):
    """Actualiza campos de pesos según los criterios seleccionados."""
    criterios = list(col_crit4.value)
    if not criterios:
        with run4_out:
            clear_output()
            print("⚠️ Seleccioná al menos un criterio.")
        return
    _actualizar_campos_pesos(criterios)

btn_generar_pesos.on_click(_on_generar_pesos_click)

# ------------------------------------------------------------
# Carga de archivo
# ------------------------------------------------------------
def _load4(change):
    with upload4_out:
        clear_output()
        if not upload4.value: return
        key = list(upload4.value.keys())[0]
        fdata = upload4.value[key]["content"]
        try:
            df = pd.read_csv(io.BytesIO(fdata)) if key.endswith(".csv") else pd.read_excel(io.BytesIO(fdata))
            L4["df_norm"] = df
            cols = list(df.columns)
            col_alt4.options = cols
            col_crit4.options = cols
            print(f"✅ {key}  |  {df.shape[0]} filas × {df.shape[1]} columnas")
            display(df.head())
            # Reiniciar pesos
            pesos_box.children = []
            pesos_widgets.clear()
        except Exception as e:
            print(f"❌ Error: {e}")

upload4.observe(_load4, names="value")

# ------------------------------------------------------------
# Actualizar visibilidad según origen de datos
# ------------------------------------------------------------
def _visibilidad_origen(change=None):
    if origen_data.value == "📂 Cargar archivo de matriz normalizada":
        upload4.layout.display = ""
        upload4_out.layout.display = ""
        col_alt4.layout.display = ""
        col_crit4.layout.display = ""
        btn_generar_pesos.layout.display = ""
        pesos_box.layout.display = ""
        # Si cambiamos a carga manual, limpiamos selección anterior
        col_alt4.options = []
        col_crit4.options = []
        pesos_box.children = []
        pesos_widgets.clear()
        L4["df_norm"] = None
    else:  # Usar L2
        upload4.layout.display = "none"
        upload4_out.layout.display = "none"
        if L2.get("df_show") is not None:
            # Cargar automáticamente de L2
            df_l2 = L2["df_show"].copy()
            L4["df_norm"] = df_l2
            cols = list(df_l2.columns)
            # La primera columna suele ser la de alternativas
            alt_col = cols[0]
            crit_cols = cols[1:] if len(cols) > 1 else []
            col_alt4.options = [alt_col]
            col_alt4.value = alt_col
            col_crit4.options = crit_cols
            col_crit4.value = crit_cols  # seleccionar todos por defecto
            col_alt4.layout.display = ""
            col_crit4.layout.display = ""
            btn_generar_pesos.layout.display = ""
            pesos_box.layout.display = ""
            # Mostrar mensaje en upload4_out (reutilizamos salida para notificar)
            with upload4_out:
                clear_output()
                print(f"✅ Matriz cargada desde L2: {df_l2.shape[0]} filas × {df_l2.shape[1]} columnas")
                display(df_l2.head())
            # Disparar creación de campos de pesos con los criterios detectados
            if crit_cols:
                _actualizar_campos_pesos(crit_cols)
        else:
            col_alt4.layout.display = "none"
            col_crit4.layout.display = "none"
            btn_generar_pesos.layout.display = "none"
            pesos_box.layout.display = "none"
            with upload4_out:
                clear_output()
                print("⚠️ No se encontró L2['df_show']. Ejecutá primero la Línea 2.")

origen_data.observe(_visibilidad_origen, names="value")

# ------------------------------------------------------------
# Ejecutar agregación
# ------------------------------------------------------------
def _run4(b):
    with run4_out:
        clear_output()
        metodo = metodo_agreg4.value
        # Obtener matriz según origen
        if L4["df_norm"] is None:
            print("❌ No hay matriz cargada. Seleccioná origen y cargá los datos.")
            return

        df_norm = L4["df_norm"]
        crit_cols = list(col_crit4.value)
        if not crit_cols:
            print("❌ Seleccioná al menos un criterio.")
            return
        alt_col = col_alt4.value
        if not alt_col or alt_col not in df_norm.columns:
            print("❌ Seleccioná una columna de alternativas válida.")
            return

        # Extraer matriz numérica de criterios
        df_c = df_norm[crit_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

        # Obtener pesos ingresados manualmente
        if not pesos_widgets:
            print("❌ Primero actualizá los campos de pesos.")
            return
        try:
            pesos_dict = {c: pesos_widgets[c].value for c in crit_cols}
        except KeyError:
            print("❌ Los criterios cambiaron. Volvé a actualizar los campos de pesos.")
            return
        if any(v < 0 for v in pesos_dict.values()):
            print("❌ Todos los pesos deben ser >= 0.")
            return
        suma_pesos = sum(pesos_dict.values())
        if suma_pesos == 0:
            print("❌ La suma de pesos no puede ser 0.")
            return

        # Normalizar pesos
        pesos_norm = np.array([pesos_dict[c] / suma_pesos for c in crit_cols])

        matriz_val = df_c.values
        alternativas = df_norm[alt_col].values
        L4["df_show"] = df_c.copy()
        L4["df_show"].insert(0, alt_col, alternativas)

        # Guardar metadatos para luego descargar
        L4["crit_cols"] = crit_cols
        L4["alternativas"] = alternativas

        # Validación para media geométrica
        matriz_transform = None
        if metodo == "Media geométrica ponderada":
            if np.any(matriz_val <= 0):
                print("❌ La media geométrica ponderada requiere valores estrictamente positivos (>0).")
                print("   Hay ceros o negativos en la matriz. Normalizá con otro método o ajustá los datos.")
                return
            u, matriz_transform = media_geometrica_ponderada(matriz_val, pesos_norm, return_matriz=True)
            L4["matriz_transform"] = matriz_transform
        else:
            u = suma_ponderada(matriz_val, pesos_norm)
            L4["matriz_transform"] = None

        # Resultados
        df_result = pd.DataFrame({"Alternativa": alternativas, "U(a)": u})
        df_result["Ranking"] = df_result["U(a)"].rank(ascending=False, method="min").astype(int)
        df_result = df_result.sort_values("Ranking")
        L4["df_agreg"] = df_result
        L4["pesos"] = pd.Series(pesos_norm, index=crit_cols)

        display(HTML(f"<b>Resultados – {metodo}</b>"))
        display(df_result.set_index("Ranking").round(4))

        # Mostrar matriz transformada (solo para media geométrica)
        if metodo == "Media geométrica ponderada" and matriz_transform is not None:
            df_transform = pd.DataFrame(matriz_transform, columns=crit_cols, index=alternativas)
            df_transform = df_transform.round(4)
            display(HTML("<br><b>📐 Matriz transformada (r_ij^w_j) – utilizada para la media geométrica ponderada</b>"))
            display(df_transform)

        # Gráfico
        fig, ax = plt.subplots(figsize=(8, max(4, len(alternativas)*0.4)))
        colores = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_result)))
        ax.barh(df_result["Alternativa"], df_result["U(a)"], color=colores, edgecolor="gray")
        ax.set_xlabel("Utilidad global U(a)", fontweight="bold")
        ax.set_title(f"Ranking – {metodo}", fontweight="bold")
        ax.invert_yaxis()
        ax.grid(axis="x", linestyle="--", alpha=0.6)
        for i, (val, alt) in enumerate(zip(df_result["U(a)"], df_result["Alternativa"])):
            ax.text(val + 0.01*df_result["U(a)"].max(), i, f"{val:.4f}", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

        dl4_btn.layout.display = ""

run4_btn.on_click(_run4)

# ------------------------------------------------------------
# Descarga de resultados
# ------------------------------------------------------------
def _download4(b):
    with dl4_out:
        clear_output()
        if L4["df_agreg"] is None:
            print("❌ Primero calculá la agregación.")
            return
        metodo = metodo_agreg4.value
        # Recuperar metadatos guardados en L4
        crit_cols = L4.get("crit_cols", [])
        alternativas = L4.get("alternativas", [])
        if not crit_cols or len(alternativas) == 0:
            print("❌ No se encontraron criterios o alternativas. Recalculá la agregación.")
            return

        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine="openpyxl") as writer:
            # Matriz normalizada
            if L4["df_show"] is not None:
                L4["df_show"].round(4).to_excel(writer, sheet_name="Matriz_normalizada", startrow=2)
                ws = writer.sheets["Matriz_normalizada"]
                ws.cell(row=1, column=1, value="MATRIZ NORMALIZADA")
            # Pesos
            if L4["pesos"] is not None:
                df_pesos = L4["pesos"].to_frame("Peso normalizado").round(4)
                df_pesos.to_excel(writer, sheet_name="Pesos", startrow=2)
                ws2 = writer.sheets["Pesos"]
                ws2.cell(row=1, column=1, value="PESOS UTILIZADOS")
            # Resultados
            L4["df_agreg"].round(4).to_excel(writer, sheet_name="Resultados", index=False, startrow=2)
            ws3 = writer.sheets["Resultados"]
            ws3.cell(row=1, column=1, value=f"RESULTADOS – {metodo} | Línea 4 – Modelos de Decisión")
            # Matriz transformada (solo para media geométrica)
            if metodo == "Media geométrica ponderada" and L4.get("matriz_transform") is not None:
                df_trans = pd.DataFrame(L4["matriz_transform"],
                                        columns=crit_cols,
                                        index=alternativas).round(4)
                df_trans.to_excel(writer, sheet_name="Matriz_transformada", startrow=2)
                ws4 = writer.sheets["Matriz_transformada"]
                ws4.cell(row=1, column=1, value="MATRIZ r_ij^w_j – Media Geométrica Ponderada")
        buf.seek(0)
        b64 = base64.b64encode(buf.getvalue()).decode()
        display(HTML(f'<a download="agregacion.xlsx" '
                     f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">'
                     f'⬇ Descargar agregacion.xlsx</a>'))

dl4_btn.on_click(_download4)

# ------------------------------------------------------------
# Mostrar interfaz completa
# ------------------------------------------------------------
display(widgets.HTML("<h3>📊 LÍNEA 4 – Agregación Multicriterio <small>Suma Ponderada · Media Geométrica Ponderada</small></h3>"))
display(_sep("Origen de los datos"))
display(origen_data)
display(_sep("Carga de archivo (si corresponde)"))
display(upload4, upload4_out)
display(_sep("Seleccionar columnas"))
display(widgets.HBox([col_alt4, col_crit4]))
display(btn_generar_pesos)
display(_sep("Pesos de cada criterio (ajustar manualmente)"))
display(pesos_box)
display(_sep("Método de agregación"))
display(metodo_agreg4)
display(_sep("Ejecutar"))
display(run4_btn, run4_out)
display(dl4_btn, dl4_out)

# Ajustar visibilidad inicial
_visibilidad_origen()